In [1]:
# %% ─────────────────────────────────────────────
# 0) Imports & global config
# ------------------------------------------------
import os, re, glob
from pathlib import Path
import numpy as np, pandas as pd, nibabel as nib
from nilearn import datasets, image
from scipy.optimize import nnls
from tqdm.notebook import tqdm
from collections import defaultdict
from scipy import stats
from statsmodels.stats.multitest import fdrcorrection
import matplotlib.pyplot as plt

In [2]:
np.random.seed(42)

In [3]:
# ---- stories to pool ---------------------------------------------------
STIMS = [ "slumlordreach", "pieman", "black", "forgot", "reachforstars", "notthefallintact"]                 # ["slumlordreach", "pieman", "black", "forgot", "reachforstars", "notthefallintact"]

# ---- trait models ------------------------------------------------------
TRAIT_SETS = {
    "all_13": [
        "Open-minded","feeling Affectionate","Attentive","Assertive",
        "feeling Gloomy","feeling Peaceful","Agreeable","Judging",
        "feeling Angry","feeling Bewildered","Impulsive",
        "Self-disciplined","Contemplating"
    ],
    "mental_8": [
        "feeling Affectionate","feeling Gloomy","feeling Peaceful",
        "feeling Angry","feeling Bewildered","Judging",
        "Contemplating","Attentive"
    ],
    "personality_5": [
        "Open-minded","Agreeable","Assertive",
        "Self-disciplined","Impulsive"
    ],
    "trait_9": [
        "Open-minded","feeling Affectionate","Attentive","Assertive",
        "Agreeable","Judging","feeling Angry","Self-disciplined","Contemplating"
    ]
}
# select which trait model to use here:
model_key = "mental_8"               # options: all_13, mental_8, personality_5, trait_9 
traits    = TRAIT_SETS[model_key]
TRAIT_SAVE = [t.replace(" ","_").replace("-","_") for t in traits]

# ---- misc hyper-params -------------------------------------------------
root_dir  = Path("/Volumes/Passport/fmriprep")
deriv_dir = root_dir / "derivatives"
shift_window      = 20
smoothing_setting = "_no_smoothing"     # options: " " or "_no_smoothing"
n_perm            = 100
PARCEL_OF_INTEREST = 68

In [4]:
# Expected # subject-run CSVs that should exist after NNLS per-parcel analysis
EXPECTED_CSV_COUNTS = {
    "slumlordreach":   17,
    "pieman":          75,
    "black":           46,
    "forgot":          46,
    "reachforstars":   17,
    "notthefallintact":54,
}

In [ ]:

# %% ─────────────────────────────────────────────
# 1) Gather all subject-level CSVs across stories
# ------------------------------------------------
df_list = []
for stim in STIMS:
    csvs = glob.glob(str(
        deriv_dir / "RSA_stats" / stim / "multi_regression" / "subject_results" /
        f"*_{stim}_multi_parcel_RSA_NNLS_{model_key}{smoothing_setting}.csv"
    ))
    
    # ── single-line hard checks ────────────────────────────────
    assert stim in EXPECTED_CSV_COUNTS, f"No expected count set for '{stim}'"
    assert len(csvs) == EXPECTED_CSV_COUNTS[stim], \
           f"[{stim}] expected {EXPECTED_CSV_COUNTS[stim]} CSVs, found {len(csvs)}"

    for fn in csvs:
        df = pd.read_csv(fn)
        df["stim"]    = stim
        df["subject"] = df["subject"] + "_" + stim            # unique ID
        df["unit"]    = df["subject"] + "_" + df["run"].astype(str)
        df_list.append(df)

all_df = pd.concat(df_list, ignore_index=True)
print("pooled CSV shape →", all_df.shape)

# %% ─────────────────────────────────────────────
# 2) Cache neural RDMs for ALL stims
# ------------------------------------------------
n_rois = 200
schaefer = datasets.fetch_atlas_schaefer_2018(
              n_rois=n_rois, yeo_networks=17, resolution_mm=2)
atlas_raw = nib.load(schaefer["maps"])
labels    = np.insert(
               [l.replace(b"17Networks_", b"").decode("utf-8")
                for l in schaefer["labels"]],
               0, "Background")

neural_cache = {}
atlas_img    = None      

for stim in STIMS:
    cleaned_root = deriv_dir / f"{stim}_cleaned"
    subs = sorted(
    s for s in os.listdir(cleaned_root) if s.startswith("sub-")
)
    for sub in subs:
        func = cleaned_root / sub / "func"
        bolds = glob.glob(str(func / f"{sub}_task-{stim}_run-*_*cleaned*.nii.gz"))
        bolds += glob.glob(str(func / f"{sub}_task-{stim}_cleaned_desc-masked_bold.nii.gz"))

        for bf in tqdm(bolds, desc=f"{stim}: caching RDMs", leave=False, disable=True):
            run = re.search(r"_run-(\d+)_", os.path.basename(bf))
            run = run.group(1) if run else "NA"
            sub_id = f"{sub}_{stim}"

            img = nib.load(bf)
        
            # ── resample Schaefer atlas to this BOLD image every time ──
            atlas_res = image.resample_to_img(atlas_raw, img, interpolation="nearest")
            atlas_dat = atlas_res.get_fdata().astype(int)

        # keep one copy of resampled atlas for later NIfTI writing
            if atlas_img is None:
                atlas_img = atlas_res
            
            bold_dat  = img.get_fdata()

            for pid in range(1, n_rois + 1):
                mask = atlas_dat == pid
                if not mask.any():
                    continue
                rdm = 1.0 - np.corrcoef(bold_dat[mask, :].T)
                neural_cache[(sub_id, run, pid)] = rdm.astype(np.float32)

print("cached RDMs →", len(neural_cache))

# %% ─────────────────────────────────────────────
# 3) Load ALL behavior RDMs once
# ------------------------------------------------
behav_rdms = {}
for stim in STIMS:
    for trait, save in zip(traits, TRAIT_SAVE):
        behav_rdms[(stim, trait)] = np.load(
            deriv_dir / "RDMs_behavior" /
            f"{stim}_{save}_RDM{smoothing_setting}.npy"
        )

# get lower-triangular indices for vectorizing RDMs
vec_idx = np.tril_indices(behav_rdms[(STIMS[0], traits[0])].shape[0], k=-1)
def vec(mat): return mat[vec_idx]


# ── Per-(subject, run, trait) shift schedules: UNIQUE across permutations ──
rng = np.random.default_rng(42)

# collect unique (subject, run) pairs from neural_cache
run_keys = sorted({(sub, run) for (sub, run, _pid) in neural_cache.keys()})

# determine n_tr per (subject, run) (check consistent across parcels)
run_ntr = {}
for (sub, run, pid), nrdm in neural_cache.items():
    ntr = nrdm.shape[0]
    prev = run_ntr.get((sub, run))
    if prev is None:
        run_ntr[(sub, run)] = ntr
    elif prev != ntr:
        raise ValueError(f"Inconsistent n_tr for {(sub, run)}: {prev} vs {ntr}")

# build NO-REPEAT shift schedules for each (subject, run, trait)
# (traits shift by different amounts)
shifts_by_run_trait = {}
for (sub, run) in run_keys:
    ntr = run_ntr[(sub, run)]
    allowed = np.arange(shift_window, ntr - shift_window, dtype=int)
    if n_perm > allowed.size:
        raise ValueError(
            f"n_perm ({n_perm}) exceeds allowed shifts ({allowed.size}) "
            f"for run {(sub, run)} with n_tr={ntr} and shift_window={shift_window}"
        )
    for t in traits:
        shifts_by_run_trait[(sub, run, t)] = rng.choice(allowed, size=n_perm, replace=False)


# %% ─────────────────────────────────────────────
# 4) Compute observed sₖ per parcel
# ------------------------------------------------
trait_cols = TRAIT_SAVE
for k in range(1, len(trait_cols)+1):
    all_df[f"s{k}"] = (np.sort(all_df[trait_cols].values, 1)[:, ::-1][:, :k].sum(1))
observed = {k : all_df.groupby("parcel_num")[f"s{k}"].mean()
            for k in range(1, len(trait_cols)+1)}

# 5) Build run-level nulls and Z-scores
# ------------------------------------------------
parcel_ids = list(observed[1].index) 
# storage: z_scores[k][pid] -> list of z values (one per run)
z_scores = {k: {pid: [] for pid in parcel_ids} for k in range(1, len(trait_cols) + 1)}

rng = np.random.default_rng(42)   

for key in tqdm(sorted(neural_cache.keys()), desc="subject-runs"):
    sub, run, pid = key
    nrdm_orig = neural_cache[key]
    stim      = sub.split("_")[-1]

    # ------- build null for this run & parcel ----------
    null_vals = {k: [] for k in range(1, len(trait_cols) + 1)}
    for i in range(n_perm):
        # leave NEURAL RDM UNCHANGED
        y = vec(nrdm_orig)

        # build design from TRAIT RDMs, each circular-shifted by its own
        # (subject, run, trait)-specific, no-repeat shift for permutation i
        X_cols = [np.ones_like(y)]
        for t in traits:
            shift_t = shifts_by_run_trait[(sub, run, t)][i]
            R = behav_rdms[(stim, t)]
            R_roll = np.roll(np.roll(R, shift_t, axis=0), shift_t, axis=1)
            X_cols.append(vec(R_roll))
        X = np.column_stack(X_cols)

        betas, _ = nnls(X, y)
        top_desc = np.sort(betas[1:])[::-1]
        for k in range(1, len(trait_cols) + 1):
            null_vals[k].append(top_desc[:k].sum())

    # ------- observed s_k for this run -----------------
    X_obs = np.column_stack([np.ones_like(vec(nrdm_orig))] +
                            [vec(behav_rdms[(stim, t)]) for t in traits])
    betas_obs, _ = nnls(X_obs, vec(nrdm_orig))
    top_desc_obs = np.sort(betas_obs[1:])[::-1]

    # ------- convert to Z and store --------------------
    for k in range(1, len(trait_cols) + 1):
        s_obs = top_desc_obs[:k].sum()
        null   = np.asarray(null_vals[k])
        p_one  = ((null >= s_obs).sum() + 1) / (n_perm + 1)

        # prevent 0 or 1 exactly
        eps = 1e-10
        p_one = np.clip(p_one, eps, 1.0 - eps)
    
        z      = stats.norm.isf(p_one)                       
        z_scores[k][pid].append(z)

# 6) One-sample t-test on Z-scores + FDR
# ------------------------------------------------

rows = []
for k in range(1, len(trait_cols) + 1):
    pvals = []
    for pid in parcel_ids:
        z_vec = z_scores[k][pid]
        if len(z_vec) == 0:
            p = 1.0; t = 0.0
        else:
            t, p_left = stats.ttest_1samp(z_vec, 0, alternative="greater")
            p = p_left   
        pvals.append(p)
        rows.append([k, pid, t, p])   

    # FDR across 200 parcels for this k
    _, p_fdr = fdrcorrection(pvals, alpha=0.05)
    for idx, pid in enumerate(parcel_ids):
        rows[(k-1)*len(parcel_ids)+idx].extend([p_fdr[idx], pvals[idx]<0.05, p_fdr[idx]<0.05])

df_perm = pd.DataFrame(rows, columns=[
            "k","parcel_num","t_value","p_uncorr",
            "p_fdr","sig_uncorr_p05","sig_fdr_p05"])
df_perm["parcel_label"] = df_perm["parcel_num"].map(
            {i:l for i,l in enumerate(labels)})

out_dir = deriv_dir / "RSA_stats" / "ALLSTIMS" / "perm_test_neural_shift"
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / f"New_ALLSTIMS_perm_test_{model_key}_allks{smoothing_setting}.csv"
df_perm.to_csv(csv_path, index=False)
print("CSV →", csv_path)



pooled CSV shape → (51000, 18)


slumlordreach: caching RDMs:   0%|          | 0/1 [00:00<?, ?it/s]

slumlordreach: caching RDMs:   0%|          | 0/1 [00:00<?, ?it/s]

slumlordreach: caching RDMs:   0%|          | 0/1 [00:00<?, ?it/s]

slumlordreach: caching RDMs:   0%|          | 0/1 [00:00<?, ?it/s]

...

notthefallintact: caching RDMs:   0%|          | 0/1 [00:00<?, ?it/s]

notthefallintact: caching RDMs:   0%|          | 0/1 [00:00<?, ?it/s]

cached RDMs → 51000


subject-runs:   0%|          | 0/51000 [00:00<?, ?it/s]

CSV → /Volumes/Passport/fmriprep/derivatives/RSA_stats/ALLSTIMS/perm_test_neural_shift/New_ALLSTIMS_perm_test_mental_8_allks_no_smoothing.csv


In [6]:
# Pick one parcel and k=1
pid = 33
z_runs = np.asarray(z_scores[1][pid])

print("mean z =", z_runs.mean(), "std =", z_runs.std())
print("runs with z > 0:", (z_runs > 0).sum(), "/", len(z_runs))

mean z = -0.7623662364617335 std = 1.8917724436705394
runs with z > 0: 86 / 255


In [7]:
for pid in [33, 68]:
    z = np.asarray(z_scores[1][pid])
    print(pid, "mean", z.mean(), " >0 runs:", (z>0).sum())

33 mean -0.7623662364617335  >0 runs: 86
68 mean -0.69233278325394  >0 runs: 90


In [8]:
# overall distribution of run z-scores (k=1)
all_z = np.concatenate([z_scores[1][pid] for pid in parcel_ids])
print("1–25–50–75–99 percentiles:",
      np.percentile(all_z, [1,25,50,75,99]))

1–25–50–75–99 percentiles: [-6.36134089 -1.41041953 -0.53296269  0.31537724  2.33007892]


In [9]:
# sign consistency within each parcel (k=1)
pos_frac = [(pid, (np.asarray(z_scores[1][pid]) > 0).mean())
            for pid in parcel_ids]
# 5 most negative-leaning parcels
print("Parcels with fewest positive runs:",
      sorted(pos_frac, key=lambda x:x[1])[:5])
# 5 most positive-leaning parcels
print("Parcels with most positive runs:",
      sorted(pos_frac, key=lambda x:x[1], reverse=True)[:5])

Parcels with fewest positive runs: [(31, 0.25882352941176473), (30, 0.2627450980392157), (168, 0.26666666666666666), (167, 0.27058823529411763), (162, 0.27450980392156865)]
Parcels with most positive runs: [(123, 0.41568627450980394), (79, 0.396078431372549), (88, 0.396078431372549), (103, 0.396078431372549), (195, 0.396078431372549)]


In [28]:
# ---- build p/s NIfTI for k = 1 -----------------------------------------
df_k1 = df_perm[df_perm.k == 1]

p_col = "p_fdr"          # <-- use FDR-corrected p-values
thr_value = 0.05         # threshold on FDR p

atlas_dat = atlas_img.get_fdata().astype(int)
shape = atlas_dat.shape

p_map  = np.zeros(shape,float); p_thr = np.full(shape,np.nan,float)
s_map  = np.zeros(shape,float); s_thr = np.zeros(shape,float)

for _,r in df_k1.iterrows():
    pid = int(r.parcel_num)
    p   = r[p_col]
    s   = r.t_value        # or r.observed_s if you want s-maps

    p_map[atlas_dat==pid] = p
    s_map[atlas_dat==pid] = s
    if p < thr_value:
        p_thr[atlas_dat==pid] = p
        s_thr[atlas_dat==pid] = s
def save(img, name): nib.save(img, out_dir/name)

save(nib.Nifti1Image(p_map,  atlas_img.affine, atlas_img.header),
     f"ALLSTIMS_perm_pmap_{model_key}_k1{smoothing_setting}.nii.gz")
save(nib.Nifti1Image(p_thr,  atlas_img.affine, atlas_img.header),
     f"ALLSTIMS_perm_pmap_{model_key}_k1_thresh05{smoothing_setting}.nii.gz")
save(nib.Nifti1Image(s_map,  atlas_img.affine, atlas_img.header),
     f"ALLSTIMS_perm_smap_{model_key}_k1{smoothing_setting}.nii.gz")
save(nib.Nifti1Image(s_thr,  atlas_img.affine, atlas_img.header),
     f"ALLSTIMS_perm_smap_{model_key}_k1_thresh05{smoothing_setting}.nii.gz")

print("✅ NIfTI maps saved to", out_dir)

✅ NIfTI maps saved to /Volumes/Passport/fmriprep/derivatives/RSA_stats/ALLSTIMS/perm_test_neural_shift
